# Instalar package plidar-roboticia
Nota: deve ser usado a versão mais recente rplidar-roboticia em vez da versão rplidar

In [ ]:

%pip3 install rplidar-roboticia

# Test 1 - Ligacao e diagnostico basico

Este teste abre a ligacao ao RPLIDAR, consulta a informacao do dispositivo e verifica o estado de saude. Destaca-se por ser o teste mais simples: confirma apenas se o LIDAR comunica corretamente antes de tentar recolher scans.

In [ ]:
from rplidar import RPLidar

PORT = '/dev/ttyACM1'
BRATE=115200
lidar = RPLidar(PORT, baudrate=BRATE)

print(lidar.get_info())
print(lidar.get_health())
print("LIDAR")

lidar.stop()
lidar.disconnect()

# Test 2 - Primeiros scans em texto

Este teste reinicia o LIDAR, espera pela estabilizacao e imprime o numero de pontos de algumas voltas completas. Destaca-se do Teste 1 porque ja recolhe medicoes reais, mas ainda sem visualizar os pontos.

In [ ]:
from rplidar import RPLidar
import time

PORT = '/dev/ttyACM1'
BRATE=115200
lidar = RPLidar(PORT, baudrate=BRATE)

print(lidar.get_info())
print(lidar.get_health())
print("LIDAR")

lidar.reset()
time.sleep(2)

for i, scan in enumerate(lidar.iter_scans()):
    print(f"Scan {i}: {len(scan)} pontos")
    
    if i >= 5:
        break

print("Disconnect from LIDAR")
lidar.stop()
lidar.disconnect()

# Test 3 - Arranque controlado com tratamento de erros

Este teste faz uma sequencia mais cuidadosa de paragem, reset, arranque do motor e leitura dos scans, incluindo `try/finally` e captura de excecoes do RPLIDAR. Destaca-se por ser mais robusto para diagnosticar problemas de buffer, motor ou comunicacao.

In [ ]:
from rplidar import RPLidar, RPLidarException
import time

PORTA = '/dev/ttyACM1'
BAUDRATE = 115200
#BAUDRATE = 256000


lidar = RPLidar(PORTA, baudrate=BAUDRATE, timeout=3)

try:
    lidar.stop()
    lidar.stop_motor()
    time.sleep(1)

    lidar.reset()
    time.sleep(2)

    print(lidar.get_info())
    print(lidar.get_health())

    lidar.start_motor()
    time.sleep(2)

    for i, scan in enumerate(lidar.iter_scans(max_buf_meas=100)):
        print("Volta", i + 1, "pontos:", len(scan))
        print(scan[:5])

except RPLidarException as e:
    print("Erro RPLidar:", e)

finally:
    try:
        lidar.stop()
        lidar.stop_motor()
    except:
        pass
    lidar.disconnect()

# Test 4 - Visualizacao cartesiana de um scan

Este teste recolhe uma unica rotacao do LIDAR e converte angulo/distancia em coordenadas X/Y para desenhar um grafico cartesiano. Destaca-se por mostrar a forma do espaco em redor do JetRacer num mapa 2D simples.

In [ ]:
from rplidar import RPLidar
import numpy as np
import matplotlib.pyplot as plt

# Grafico Cartesiano

PORTA = '/dev/ttyACM1'
BAUDRATE = 115200
lidar = RPLidar(PORTA, baudrate=BAUDRATE, timeout=3)

try:
    # Obtém uma única rotação completa
    scan = next(lidar.iter_scans())

    x = []
    y = []

    for quality, angle, distance in scan:

        # mm -> metros
        d = distance / 1000.0

        # graus -> radianos
        a = np.radians(angle)

        x.append(d * np.cos(a))
        y.append(d * np.sin(a))

    plt.figure(figsize=(8,8))
    plt.scatter(x, y, s=5)

    # posição do robô
    plt.scatter([0], [0], marker='x', s=100)

    plt.axis('equal')
    plt.grid(True)

    plt.xlabel('X (m)')
    plt.ylabel('Y (m)')
    plt.title('RPLIDAR A1 - Scan 360º')

    plt.show()

finally:
    lidar.stop()
    lidar.stop_motor()
    lidar.disconnect()

# Test 5 - Visualizacao polar de um scan

Este teste tambem usa uma unica rotacao, mas apresenta os pontos num grafico polar, mantendo diretamente a relacao entre angulo e distancia. Destaca-se do Teste 4 por mostrar os dados no formato mais natural para um sensor rotativo.

In [ ]:
from rplidar import RPLidar
import numpy as np
import matplotlib.pyplot as plt

# Grafico Polar

PORTA = '/dev/ttyACM1'
BAUDRATE = 115200
lidar = RPLidar(PORTA, baudrate=BAUDRATE, timeout=3)

try:

    scan = next(lidar.iter_scans())

    angles = []
    distances = []

    for _, angle, distance in scan:

        angles.append(np.radians(angle))
        # mm -> metros
        distances.append(distance / 1000.0) 

    fig = plt.figure(figsize=(8,8))

    ax = fig.add_subplot(111, polar=True)

    ax.scatter(
        angles,
        distances,
        s=5
    )

    ax.set_title("RPLIDAR A1")

    plt.show()

finally:
    lidar.stop()
    lidar.stop_motor()
    lidar.disconnect()

# Test 6 - Visualizacao continua em tempo real

Este teste atualiza continuamente um grafico cartesiano com scans sucessivos, ignorando algumas voltas para evitar excesso de dados no buffer. Destaca-se por ser o teste mais proximo de uma aplicacao em tempo real, permitindo observar mudancas no ambiente enquanto o LIDAR esta a funcionar.

In [ ]:
from rplidar import RPLidar
import numpy as np
import matplotlib.pyplot as plt

# Visualização continua

PORTA = '/dev/ttyACM1'
BAUDRATE = 115200
lidar = RPLidar(PORTA, baudrate=BAUDRATE, timeout=3)

plt.ion()

fig, ax = plt.subplots(figsize=(8,8))

scatter = ax.scatter([], [], s=3)

ax.set_xlim(-6, 6)
ax.set_ylim(-6, 6)
ax.set_aspect('equal')
ax.grid(True)

try:

    for i, scan in enumerate(lidar.iter_scans()):
        
        # Avoid: Too many bytes in the input buffer...
        if i % 3 != 0:
            continue

        pontos = []

        for _, angle, distance in scan:

            if distance <= 0:
                continue

            d = distance / 1000.0
            theta = np.radians(angle)

            x = d * np.cos(theta)
            y = d * np.sin(theta)

            pontos.append([x, y])

        if len(pontos) > 0:

            pontos = np.array(pontos)

            scatter.set_offsets(pontos)

            ax.set_title(
                f"{len(scan)} medições"
            )

            fig.canvas.draw()
            fig.canvas.flush_events()

except KeyboardInterrupt:
    pass

finally:

    lidar.stop()
    lidar.stop_motor()
    lidar.disconnect()
    
